|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/deeplearning_x/?couponCode=202508" target="_blank">A deep understanding of deep learning</a></h1>|
|-|:-:|
|<h2>Section:</h2>|<h1>Overfitting, cross-validation, regularization<h1>|
|<h2>Lecture:</h2>|<h1><b>Cross-validation -- manual separation<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/deeplearning_x/?couponCode=202508" target="_blank">udemy.com/course/deeplearning_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [1]:
%matplotlib widget

# import libraries
import torch
import torch.nn as nn
import numpy as np

In [2]:
# import dataset
import pandas as pd
iris = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')


# convert from pandas dataframe to tensor
data = torch.tensor( iris[iris.columns[0:4]].values ).float()

# transform species to number
labels = torch.zeros(len(data), dtype=torch.long)
# labels[iris.species=='setosa'] = 0 # don't need!
labels[iris.species=='versicolor'] = 1
labels[iris.species=='virginica'] = 2

# Separate data into train and test

In [3]:
#  (no devset here)

# how many training examples
propTraining = .8 # in proportion, not percent
nTraining = int(len(labels)*propTraining)

# initialize a boolean vector to select data and labels
traintestBool = np.zeros(len(labels),dtype=bool)

# is this the correct way to select samples?  No. The data will not be randomly shuffled.
# traintestBool[range(nTraining)] = True

# this is better, but why?
items2use4train = np.random.choice(range(len(labels)),nTraining,replace=False)
traintestBool[items2use4train] = True

traintestBool

array([ True, False,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True, False,  True,  True,  True, False, False,
       False, False, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False, False,  True,  True,  True,  True,  True, False,
        True, False, False,  True,  True,  True, False, False,  True,
       False,  True, False, False,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False,  True,  True,  True,  True,
       False, False,  True,  True, False,  True, False,  True, False,
       False,  True,

In [4]:
# 1) Randomly assigning data samples to be in the train vs test phase produced a statistical balance, but it was
#    not perfect. Write an algorithm that will guarantee a balance of flower types while also randomly assigning
#    samples to be in train vs. test.

# Get all unique classes:
unique_classes = torch.unique(labels)

# Count classes of each type:
class_counts = {cls.item(): (labels == cls).sum().item() for cls in unique_classes}
# class_counts

# Randomly select indexes of each class for trainining and testing set in proprotion of .8 to .2:
train_indexes = []
test_indexes = []
for cls in unique_classes:
    cls_indexes = torch.where(labels == cls)[0].tolist()
    n_cls_training = int(len(cls_indexes) * propTraining)
    selected_train_indexes = np.random.choice(cls_indexes, n_cls_training, replace=False).tolist()
    train_indexes.extend(selected_train_indexes)
    test_indexes.extend([idx for idx in cls_indexes if idx not in selected_train_indexes])
    
print(f"Train indexes: {len(train_indexes)}")
print(f"Test indexes: {len(test_indexes)}")

# Count number of each class in training and testing set:
train_class_counts = {cls.item(): sum(1 for idx in train_indexes if labels[idx] == cls) for cls in unique_classes}
test_class_counts = {cls.item(): sum(1 for idx in test_indexes if labels[idx] == cls) for cls in unique_classes}

print(f"Training set class counts: {train_class_counts}")
print(f"Testing set class counts: {test_class_counts}")

Train indexes: 120
Test indexes: 30
Training set class counts: {0: 40, 1: 40, 2: 40}
Testing set class counts: {0: 10, 1: 10, 2: 10}


In [5]:
# test whether it's balanced
print('Average of full data:')
print( torch.mean(labels.float()) ) # =1 by definition
print(' ')

print('Average of training data:')
print( torch.mean(labels[traintestBool].float()) ) # should be 1...
print(' ')

print('Average of test data:')
print( torch.mean(labels[~traintestBool].float()) ) # should also be 1...

Average of full data:
tensor(1.)
 
Average of training data:
tensor(1.0167)
 
Average of test data:
tensor(0.9333)


In [6]:
# create the ANN model

# model architecture
ANNiris = nn.Sequential(
    nn.Linear(4,64),   # input layer
    nn.ReLU(),         # activation unit
    nn.Linear(64,64),  # hidden layer
    nn.ReLU(),         # activation unit
    nn.Linear(64,3),   # output units
      )

# loss function
lossfun = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.SGD(ANNiris.parameters(),lr=.01)

In [7]:
# entire dataset
print( data.shape )

# training set
print( data[traintestBool,:].shape )

# test set
print( data[~traintestBool,:].shape )

torch.Size([150, 4])
torch.Size([120, 4])
torch.Size([30, 4])


# Train and test the model

In [8]:
# train the model

numepochs = 1000

# initialize losses
losses = torch.zeros(numepochs)
ongoingAcc = []

# loop over epochs
for epochi in range(numepochs):

  # forward pass
  yHat = ANNiris(data[traintestBool,:])

  # compute accuracy (note: denser than previous code!)
  ongoingAcc.append( 100*torch.mean(
              (torch.argmax(yHat,axis=1) == labels[traintestBool]).float()) )

  # compute loss
  loss = lossfun(yHat,labels[traintestBool])
  losses[epochi] = loss

  # backprop
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [9]:
# compute train and test accuracies

# final forward pass USING TRAINING DATA
predictions = ANNiris(data[traintestBool,:])
trainacc = 100*torch.mean((torch.argmax(predictions,axis=1) == labels[traintestBool]).float())


# final forward pass USING TEST DATA!
predictions = ANNiris(data[~traintestBool,:])
testacc = 100*torch.mean((torch.argmax(predictions,axis=1) == labels[~traintestBool]).float())

In [10]:
# report accuracies

print('Final TRAIN accuracy: %g%%' %trainacc)
print('Final TEST accuracy:  %g%%' %testacc)

Final TRAIN accuracy: 98.3333%
Final TEST accuracy:  96.6667%


In [ ]:
# 2) Revert the code to its original form -- with the strong imbalance in flower types. Then train the model. What are
#    the train and test accuracies? Compute the accuracy separately for each type of flower to see whether the model
#    learned some categories, or whether it performed equally on all three categories. Are you surprised at the results?

# Compute train and test accuracies per class
for cls in unique_classes:
    cls_train_acc = 100 * torch.mean(
        (torch.argmax(ANNiris(data[traintestBool,:]), axis=1)[labels[traintestBool] == cls] == cls).float()
    )
    cls_test_acc = 100 * torch.mean(
        (torch.argmax(ANNiris(data[~traintestBool,:]), axis=1)[labels[~traintestBool] == cls] == cls).float()
    )
    print(f'Class {cls.item()} - TRAIN accuracy: {cls_train_acc}%, TEST accuracy: {cls_test_acc}%')

Class 0 - TRAIN accuracy: 100.0%, TEST accuracy: 100.0%
Class 1 - TRAIN accuracy: 94.73684692382812%, TEST accuracy: 91.66667175292969%
Class 2 - TRAIN accuracy: 100.0%, TEST accuracy: 100.0%


In [11]:
# normally also inspect losses and accuracy by epoch, etc etc etc.

# Additional explorations

In [ ]:
# 1) Randomly assigning data samples to be in the train vs test phase produced a statistical balance, but it was
#    not perfect. Write an algorithm that will guarantee a balance of flower types while also randomly assigning
#    samples to be in train vs. test.
#
# 2) Revert the code to its original form -- with the strong imbalance in flower types. Then train the model. What are
#    the train and test accuracies? Compute the accuracy separately for each type of flower to see whether the model
#    learned some categories, or whether it performed equally on all three categories. Are you surprised at the results?
# 
# The results are not surprising because class 1 is much more difficult to separate also as a human, while keeping model generalization.
#